# Notebook 2 — Open Banking Model: Transaction Data Only

**Project:** Open Banking Credit Risk Model  
**Author:** Chetan Prakash | MSc FinTech, University of Glasgow  

---

## Objective
Can we predict credit default using **only 90 days of transaction behaviour** — with zero credit history?

This is the exact scenario faced by:
- BNPL lenders (no CIBIL access)
- Neobanks onboarding first-time borrowers
- Fintechs operating under India's Account Aggregator framework

## Feature Engineering — 56 Behavioural Features

| Group | Features |
|-------|----------|
| **Volume** | avg, total, std, max, min, median spend; txn/day |
| **24 Category Ratios** | Groceries, Dining, Gambling, Salary Credit, Utilities, Rent, etc. |
| **11 Category Totals** | Total spend in high-signal categories |
| **5 Method Ratios** | Credit Card, Debit Card, Cash, Direct Debit, Bank Transfer |
| **Composite Scores** | `high_risk_ratio`, `stability_ratio`, `essential_ratio` |
| **Behavioural Timing** | `late_night_ratio`, `weekend_spend_ratio` |
| **Income Signals** | `has_salary`, `cash_ratio` |

## Data
| File | Role | Shape |
|------|------|-------|
| `old_cust_transactions.csv` | Training | 1,406,004 rows |
| `new_cust_transactions.csv` | Test | 154,232 rows |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
)
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
# Load raw transactions
old_credit = pd.read_csv('../data/old_cust_credit.csv')
new_credit = pd.read_csv('../data/new_cust_credit.csv')
old_txn = pd.read_csv('../data/old_cust_transactions.csv', parse_dates=['timestamp'])
new_txn = pd.read_csv('../data/new_cust_transactions.csv', parse_dates=['timestamp'])

print(f'Old transactions : {len(old_txn):,} rows | {old_txn["account_id"].nunique()} accounts')
print(f'New transactions : {len(new_txn):,} rows | {new_txn["account_id"].nunique()} accounts')
print(f'\nCategories ({old_txn["category"].nunique()}):')
print(old_txn['category'].value_counts().head(10).to_string())

## 2. Feature Engineering

In [ ]:
ALL_CATS = [
    'Groceries','Dining','Online Shopping','Entertainment','Bars/Nightlife',
    'ATM Withdrawal','Gym/Sports','Clothing','Travel','Public Transit',
    'Fuel','Transfer Out','Utilities','Pharmacy','Savings Transfer',
    'Healthcare','Gambling','Electronics','Education','Refund',
    'Fees/Penalties','Salary Credit','Insurance','Rent'
]
ALL_METHODS = ['Credit Card','Debit Card','Cash Deposit','Direct Debit','Bank Transfer']
HIGH_RISK   = ['Gambling','Bars/Nightlife','ATM Withdrawal','Fees/Penalties','Transfer Out']
STABLE      = ['Utilities','Insurance','Rent','Savings Transfer','Salary Credit']
ESSENTIAL   = ['Groceries','Pharmacy','Healthcare','Public Transit','Fuel']

def engineer_features(txn_df, label_map=None):
    """
    Aggregate raw transaction rows into account-level behavioural features.
    Applied identically to training and test data.
    """
    df = txn_df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date']    = df['timestamp'].dt.date
    df['hour']    = df['timestamp'].dt.hour
    df['weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(int)

    df = pd.get_dummies(df, columns=['category'], prefix='cat')
    df = pd.get_dummies(df, columns=['method'],   prefix='mth')
    for c in ALL_CATS:
        col = f"cat_{c.replace('/','_').replace(' ','_')}"
        if col not in df.columns: df[col] = 0
    for m in ALL_METHODS:
        col = f"mth_{m.replace(' ','_')}"
        if col not in df.columns: df[col] = 0

    grp = df.groupby('account_id')
    feats = grp['amount'].agg(
        avg_txn_amount='mean', total_spend='sum', txn_count='count',
        std_txn_amount='std', max_txn_amount='max',
        min_txn_amount='min', median_txn_amount='median'
    ).reset_index()
    feats = feats.merge(grp['date'].nunique().reset_index(name='active_days'), on='account_id')
    feats['txn_per_day'] = feats['txn_count'] / feats['active_days'].replace(0,1)

    for m in ALL_METHODS:
        col = f"mth_{m.replace(' ','_')}"
        if col in df.columns:
            feats = feats.merge(grp[col].mean().reset_index(
                name=f"ratio_{m.replace(' ','_').lower()}"), on='account_id')

    for c in ALL_CATS:
        col = f"cat_{c.replace('/','_').replace(' ','_')}"
        if col in df.columns:
            nm = f"ratio_{c.replace('/','_').replace(' ','_').lower()}"
            feats = feats.merge(grp[col].mean().reset_index(name=nm), on='account_id')

    for c in ['Gambling','Fees/Penalties','Bars/Nightlife','ATM Withdrawal',
               'Salary Credit','Savings Transfer','Utilities','Rent',
               'Online Shopping','Travel','Transfer Out']:
        col = f"cat_{c.replace('/','_').replace(' ','_')}"
        if col in df.columns:
            nm  = f"total_{c.replace('/','_').replace(' ','_').lower()}"
            tmp = grp.apply(lambda x: (x['amount'] * x[col]).sum()).reset_index(name=nm)
            feats = feats.merge(tmp, on='account_id')

    hr = [f"ratio_{c.replace('/','_').replace(' ','_').lower()}" for c in HIGH_RISK
          if f"ratio_{c.replace('/','_').replace(' ','_').lower()}" in feats.columns]
    st = [f"ratio_{c.replace('/','_').replace(' ','_').lower()}" for c in STABLE
          if f"ratio_{c.replace('/','_').replace(' ','_').lower()}" in feats.columns]
    es = [f"ratio_{c.replace('/','_').replace(' ','_').lower()}" for c in ESSENTIAL
          if f"ratio_{c.replace('/','_').replace(' ','_').lower()}" in feats.columns]
    feats['high_risk_ratio']  = feats[hr].sum(axis=1)
    feats['stability_ratio']  = feats[st].sum(axis=1)
    feats['essential_ratio']  = feats[es].sum(axis=1)

    feats = feats.merge(
        grp.apply(lambda x: ((x['hour']<6)|(x['hour']>=22)).mean()).reset_index(name='late_night_ratio'),
        on='account_id')
    feats = feats.merge(
        grp.apply(lambda x: x.loc[x['weekend']==1,'amount'].sum()/x['amount'].sum()
                  if x['amount'].sum()>0 else 0).reset_index(name='weekend_spend_ratio'),
        on='account_id')

    sal = 'cat_Salary_Credit'
    feats['has_salary'] = grp[sal].max().reset_index()[sal] if sal in df.columns else 0
    csh = 'mth_Cash_Deposit'
    feats['cash_ratio'] = grp[csh].mean().reset_index()[csh] if csh in df.columns else 0

    if label_map:
        feats['credit_defaults'] = feats['account_id'].map(label_map)
    return feats


old_label_map = dict(zip(old_credit['account_id'], old_credit['credit_defaults']))
new_label_map = dict(zip(new_credit['account_id'], new_credit['credit_defaults']))

print('Engineering training features (1.4M rows)...')
train_feats = engineer_features(old_txn, label_map=old_label_map)
print(f'  Done: {train_feats.shape}')

print('Engineering test features (154K rows)...')
test_feats  = engineer_features(new_txn, label_map=new_label_map)
print(f'  Done: {test_feats.shape}')

In [ ]:
# EDA: default rate by key behavioural signals
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

features_to_plot = ['high_risk_ratio', 'stability_ratio', 'total_gambling']
titles = ['High-Risk Spend Ratio', 'Stability Ratio (Utilities/Rent/Savings)', 'Total Gambling Spend']

for ax, feat, title in zip(axes, features_to_plot, titles):
    for label, color, name in [(0,'steelblue','No Default'),(1,'tomato','Default')]:
        vals = train_feats[train_feats['credit_defaults']==label][feat].dropna()
        ax.hist(vals, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value'); ax.legend()

plt.suptitle('Key Behavioural Features by Default Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Train & Evaluate

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

X2  = train_feats.drop(columns=['account_id','credit_defaults'])
y2  = train_feats['credit_defaults']
X2t = test_feats.drop(columns=['account_id','credit_defaults'])
y2t = test_feats['credit_defaults']
X2t = X2t.reindex(columns=X2.columns, fill_value=0)

mask_tr = X2.notna().all(axis=1); X2,y2   = X2[mask_tr],y2[mask_tr]
mask_te = X2t.notna().all(axis=1); X2t,y2t = X2t[mask_te],y2t[mask_te]

X_tr,X_val,y_tr,y_val = train_test_split(X2,y2,test_size=0.2,random_state=SEED,stratify=y2)
scaler = StandardScaler()
Xtr_sc=scaler.fit_transform(X_tr); Xval_sc=scaler.transform(X_val); Xte_sc=scaler.transform(X2t)

# Logistic Regression
lr = LogisticRegression(penalty='l2', solver='liblinear', max_iter=3000,
                         random_state=SEED, class_weight='balanced')
lr.fit(Xtr_sc, y_tr)
cv_lr = cross_val_score(lr, Xtr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'LR  CV AUC: {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}')

# XGBoost
sp = (y_tr==0).sum()/(y_tr==1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=sp, eval_metric='logloss',
    random_state=SEED, tree_method='hist')
xgb_model.fit(Xtr_sc, y_tr, eval_set=[(Xval_sc,y_val)], verbose=False)
cv_xgb = cross_val_score(xgb_model, Xtr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'XGB CV AUC: {cv_xgb.mean():.4f} +/- {cv_xgb.std():.4f}')

In [ ]:
lr_pred  = lr.predict(Xte_sc);          lr_prob  = lr.predict_proba(Xte_sc)[:,1]
xgb_pred = xgb_model.predict(Xte_sc);   xgb_prob = xgb_model.predict_proba(Xte_sc)[:,1]

for name, pred, prob in [('Logistic Regression', lr_pred, lr_prob),
                          ('XGBoost',            xgb_pred, xgb_prob)]:
    print(f'\n--- {name} ---')
    print(f'  AUC      : {roc_auc_score(y2t, prob):.4f}')
    print(f'  Accuracy : {accuracy_score(y2t, pred):.4f}')
    print(f'  Brier    : {brier_score_loss(y2t, prob):.4f}')
    cm = confusion_matrix(y2t, pred)
    fn, tp = cm[1,0], cm[1,1]
    print(f'  Defaulters caught: {tp}/{fn+tp} ({tp/(fn+tp)*100:.0f}%)')
    print(classification_report(y2t, pred, target_names=['No Default','Default']))

In [ ]:
# ROC + Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
for prob, label, color in [(lr_prob,'Logistic Regression','steelblue'),
                             (xgb_prob,'XGBoost','darkorange')]:
    fpr, tpr, _ = roc_curve(y2t, prob)
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f'{label} (AUC={roc_auc_score(y2t,prob):.3f})')
ax.plot([0,1],[0,1],'--',color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — Open Banking\n(Transaction Data Only)', fontweight='bold')
ax.legend(loc='lower right')

for ax, pred, title, cmap in zip(axes[1:],
    [lr_pred, xgb_pred],['Logistic Regression','XGBoost'],['Blues','Oranges']):
    ConfusionMatrixDisplay(confusion_matrix(y2t, pred),
        display_labels=['No Default','Default']).plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'{title}\nOpen Banking', fontweight='bold')

plt.suptitle('Notebook 2 — Open Banking Model (Transaction Data Only)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/02_openbanking_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(xgb_model.feature_importances_, index=X_tr.columns)
top15 = importances.nlargest(15).sort_values()

plt.figure(figsize=(10, 6))
top15.plot(kind='barh', color='darkorange', edgecolor='white')
plt.title('XGBoost — Top 15 Feature Importances\n(Open Banking Model)', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Results

| Metric | Logistic Regression | XGBoost |
|--------|--------------------|---------|
| **Test AUC** | 0.9112 | 0.8992 |
| **Accuracy** | 84.4% | 83.0% |
| **Brier Score** | 0.1166 | 0.1218 |
| **Defaulters Caught** | 84% | 74% |

**Key Finding:** Transaction data alone achieves AUC 0.91 — a +28% improvement over the traditional bureau baseline (0.71).  
90 days of spending behaviour is a stronger default predictor than years of credit history for new customers.

---
**Next:** Notebook 3 combines both data sources for the hybrid model.